In [ ]:
#| default_exp historical

In [ ]:
#| export
import pandas as pd
import numpy as np
import scipy.io as sio
import rasterio
import requests
import IPython
import matplotlib.pyplot as plt
from fastcore.test import test_eq
import datetime
#from geoget.download import run_all
import banet.nrt
from banet.core import filter_files, ls, Path, InOutPath, ProjectPath
from banet.geo import Region
from banet.data import *
from banet.predict import predict_time
from fire_split.core import split_fires, save_data, to_polygon
Path.ls = ls

ModuleNotFoundError: No module named 'geoget'

In [ ]:
#| include: false
from nbdev.showdoc import show_doc
from nbdev import nbdev_export
from IPython.core.debugger import set_trace

In [ ]:
#| export
class RunManager(banet.nrt.RunManager):
    def __init__(self, project_path:ProjectPath, region:str, times:pd.DatetimeIndex,
                 product:str='VIIRS750'):
        """
        project_path: banet.core.ProjectPath object
        region: name of the region
        times: dates for the first day of month for each month to use
        product: VIIRS750 or VIIRS375
        """
        self.path    = project_path
        self.times   = self.init_times(times)
        self.product = product
        self.region  = region
        
    def init_times(self, times):
        tstart = times[0] - pd.Timedelta(days=15)
        tstart = pd.Timestamp(f'{tstart.year}-{tstart.month}-01')
        tend = times[-1] + pd.Timedelta(days=75)
        tend = pd.Timestamp(f'{tend.year}-{tend.month}-01') - pd.Timedelta(days=1)
        return pd.date_range(tstart, tend, freq='D')
        
    def check_data(self):
        "Check existing and missing files in dataset folder."
        times = self.times
        files, missing_files = [], []
        for t in times:
            tstr = t.strftime('%Y%m%d')
            file = self.path.dataset/f'{self.product}{self.region}_{tstr}.nc'
            if file.is_file():
                files.append(file)
            else:
                missing_files.append(file)
        return {'files': files, 'missing_files': missing_files}
    
    def get_download_dates(self):
        "Find for which new dates the files need to be downloaded."
        files = self.check_data()['files']
        missing_add = self.check_data()['missing_files']
        if len(files) == 0: 
            start = self.times[0]
            end = self.times[-1].strftime('%Y-%m-%d 23:59:59')
        else:
            #start = pd.Timestamp(files[-1].stem.split('_')[-1])+pd.Timedelta(days=1)
            start = pd.Timestamp(missing_add[0].stem.split('_')[-1])
            end = pd.Timestamp(missing_add[-1].stem.split('_')[-1])
            start = start.strftime('%Y-%m-%d 00:00:00')
            end = end.strftime('%Y-%m-%d 23:59:59')
        #end = self.times[-1].strftime('%Y-%m-%d 23:59:59')
        return start, end
        
    def download_viirs(self, maxOrderSize=[1800, 1200]):
        pass
    
    def get_preds(self, weight_files:list, threshold=0.5, save=True, max_size=2000,
                  filename='data', check_file=False, verbose=False):
        "Computes BA-Net predictions ensembling the models in the weight_files list."
        local_files = self.init_model_weights(weight_files)
        iop = InOutPath(self.path.dataset, self.path.outputs, mkdir=False)
        region = self.R.new()
        predict_time(iop, self.times, local_files, region, threshold=threshold,
                     save=save, max_size=max_size, product=self.product, output=filename,
                      check_file=check_file, verbose=verbose)

In [ ]:
show_doc(RunManager.preprocess_dataset)
show_doc(RunManager.init_model_weights)
show_doc(RunManager.get_preds)

<h4 id="RunManager.preprocess_dataset" class="doc_header"><code>RunManager.preprocess_dataset</code><a href="https://github.com/mnpinto/banet/tree/master/banet/nrt.py#L157" class="source_link" style="float:right">[source]</a></h4>

> <code>RunManager.preprocess_dataset</code>(**`max_size`**=*`None`*, **`max_workers`**=*`1`*)



<h4 id="RunManager.init_model_weights" class="doc_header"><code>RunManager.init_model_weights</code><a href="https://github.com/mnpinto/banet/tree/master/banet/nrt.py#L164" class="source_link" style="float:right">[source]</a></h4>

> <code>RunManager.init_model_weights</code>(**`weight_files`**:`list`)

Downloads model weights if they don't exist yet on config directory.

<h4 id="RunManager.get_preds" class="doc_header"><code>RunManager.get_preds</code><a href="__main__.py#L68" class="source_link" style="float:right">[source]</a></h4>

> <code>RunManager.get_preds</code>(**`weight_files`**:`list`, **`threshold`**=*`0.5`*, **`save`**=*`True`*, **`max_size`**=*`2000`*, **`filename`**=*`'data'`*, **`check_file`**=*`False`*, **`verbose`**=*`False`*)

Computes BA-Net predictions ensembling the models in the weight_files list.

Running all processes looks like this:

```python
region = 'PT'
paths = ProjectPath('../hide/historical_test')
weight_files = ['banetv0.20-val2017-fold0.pth']
times = pd.date_range('2017-06-01', '2017-10-01', freq='MS')
Region(region, [-10, 36, -6, 44], 0.001).export(paths.config/f'R_{region}.json')
manager = RunManager(paths, region, times, product='VIIRS375')
manager.download_viirs()
# Save hotspots{region}.csv file in hotspots folder
manager.preprocess_dataset()
manager.get_preds(weight_files, threshold=0.01, filename=f'ba100m_{region}{times[0].year}')
manager.postprocess(filename=f'ba100m', threshold=0.5, area_epsg=3763)
```


In [ ]:
#| include: false
nbdev_export()

Converted 00_core.ipynb.
Converted 01_geo.ipynb.
Converted 02_data.ipynb.
Converted 03_models.ipynb.
Converted 04_predict.ipynb.
Converted 04b_nrt.ipynb.
Converted 04c_historical.ipynb.
Converted 05_train.ipynb.
Converted 06_cli.ipynb.
Converted 07_web.ipynb.
Converted index.ipynb.
Converted tutorial.australia2020.ipynb.
Converted tutorial.australia2020_100m.ipynb.
